# ACTUNEO tour

A walk through what ACTUNEO does: Zimbabwe mortality tables, life contingencies, finance, run-off triangles, reserving diagnostics, and IFRS 17 under the premium allocation approach.

Run the cells from top to bottom. ACTUNEO is in beta: read the Status and Limitations page in the documentation before relying on any result.

## 1. Install and check

Run the install once, then restart the kernel.

In [ ]:
# %pip install --upgrade actuneo openpyxl matplotlib

import numpy as np
import pandas as pd
import actuneo

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
print(actuneo.__version__)

## 2. Zimbabwe 2023 mortality tables

In [ ]:
from actuneo import zimbabwe
from actuneo.mortality import MortalityTable

zimbabwe.mortality_tables()

In [ ]:
mt = MortalityTable.from_zimbabwe_2023("male_assured_lives")
print(mt)
print(f"q40 = {mt.qx(40):.6f}")
print(f"Life expectancy at 40: {mt.life_expectancy(40):.1f} years")
print(f"20-year survival from 40: {mt.npx(40, 20):.4f}")
mt.to_dataframe().head()

In [ ]:
print("1,000,000 ZWL in ZiG:", round(zimbabwe.zwl_to_zwg(1_000_000), 4))
# Minimum capital in USD as reported for SI 67 of 2025 (secondary source)
print("Non-life minimum capital:", zimbabwe.minimum_capital("non-life"))

## 3. Your own table and life contingencies

In [ ]:
from actuneo.mortality import SurvivalFunctions, CommutationFunctions

ages = np.arange(20, 111)
qx = np.minimum(0.0005 * 1.09 ** (ages - 20), 1.0)
qx[-1] = 1.0                      # close the table at the last age
table = MortalityTable(ages, qx, name="Example Table")

sf = SurvivalFunctions(table, interest_rate=0.05)
print(f"10p30        = {sf.npx(30, 10):.5f}")
print(f"A30          = {sf.assurance(30):.5f}")
print(f"A30:20 term  = {sf.assurance(30, 20):.5f}")
print(f"annuity-due  = {sf.annuity_due(30):.4f}")
print(f"a30:20       = {sf.annuity_immediate(30, 20):.4f}")
print(f"20E30        = {sf.pure_endowment(30, 20):.5f}")

CommutationFunctions(table, interest_rate=0.05).to_dataframe().head()

## 4. Life insurance and annuities

In [ ]:
from actuneo.life import LifeAssurance, Annuities

la = LifeAssurance(table, interest_rate=0.05)
sum_assured = 100_000
print(f"Whole life single premium at 30:  {sum_assured * la.whole_life_assurance(30):,.2f}")
print(f"20-year term annual premium:      {sum_assured * la.net_annual_premium(30, 20, 'term'):,.2f}")
print(f"20-year endowment annual premium: {sum_assured * la.net_annual_premium(30, 20, 'endowment'):,.2f}")
print(f"Endowment reserve at duration 10: {sum_assured * la.reserve_endowment(30, 20, 10):,.2f}")
print(f"Joint life assurance (40, 45):    {la.joint_life_assurance(40, 45):.5f}")

ann = Annuities(table, interest_rate=0.05)
pension = 12_000
print(f"Pension from 65, in arrears:      {ann.life_annuity_immediate(65, pension):,.2f}")
print(f"Pension from 65, monthly advance: {ann.monthly_life_annuity(65, pension):,.2f}")
print(f"With 5 years guaranteed:          {ann.guaranteed_annuity(65, 5, pension):,.2f}")
print(f"Deferred from age 45:             {ann.deferred_life_annuity(45, 20, pension):,.2f}")
print(f"Spouse's reversion (65, 60):      {ann.reversionary_annuity(65, 60, pension):,.2f}")

## 5. Finance

Rates are effective annual rates unless stated; annuity functions value 1 per year.

In [ ]:
from actuneo.finance import (
    InterestRate, ForceOfInterest, annuity, accumulated_annuity, increasing_annuity,
    Cashflows, Loan, annual_percentage_rate, Bond, par_yield, spot_rates_from_forwards,
    discounted_mean_term, immunisation_check,
)

rate = InterestRate.from_nominal(0.12, 12)        # 12% pa convertible monthly
print(f"effective {rate.i:.6f}, d {rate.d:.6f}, force {rate.delta:.6f}")

force = ForceOfInterest([(6, [0.04]), (None, [0.2, -0.02])])
print(f"400 at time 3 accumulates to {force.accumulate(400, 3, 8):.2f} at time 8")

print(f"a(20) at 10%:        {annuity(20, 0.10):.4f}")
print(f"monthly in advance:  {annuity(20, 0.10, p=12, timing='advance'):.4f}")
print(f"s(10) at 7%:         {accumulated_annuity(10, 0.07):.4f}")
print(f"(Ia)(25) at 6%:      {increasing_annuity(25, 0.06):.4f}")

In [ ]:
# Project appraisal
project = Cashflows([0], [-25_000]).add_continuous(0, 5, 8_000).add(6, -5_000)
print(f"NPV at 10%:         {project.net_present_value(0.10):,.2f}")
print(f"IRR:                {project.internal_rate_of_return():.2%}")
print(f"Discounted payback: {project.discounted_payback_period(0.10):.2f} years")

# Loan schedule
loan = Loan(50_000, 0.08, 10, payments_per_year=12)
print(f"Monthly repayment:  {loan.payment:.2f}")
print(f"APR:                {annual_percentage_rate(7500, 368.75, 24):.2%}")
loan.schedule().head()

In [ ]:
# Bonds, term structure and immunisation
bond = Bond(0.06, 13, frequency=2)
print(f"Price for 5% net yield, 40% income tax: {bond.price(0.05, income_tax=0.40):.4f}")
print(f"Gross redemption yield at 95:           {bond.redemption_yield(95):.3%}")

spots = spot_rates_from_forwards([0.06, 0.065, 0.07])
print(f"3-year par yield:     {par_yield(spots):.4%}")
print(f"Discounted mean term: {discounted_mean_term([10, 10, 110], [1, 2, 3], 0.08):.2f}")
check = immunisation_check([7.404, 31.834], [2, 25], [10, 20], [10, 15], 0.07, tolerance=1e-3)
print("Immunised:", check["immunised"])

## 6. Run-off triangles and reserving

In [ ]:
from actuneo.loss_reserving import Triangle, ChainLadder, MackChainLadder, load_raa, load_genins

raa = load_raa()                  # a well-known example triangle
print(raa)
raa.age_to_age().round(3)

In [ ]:
cl = ChainLadder(raa)
print(cl.factors.round(4))
cl.summary().round(0)

In [ ]:
mack = MackChainLadder(raa, est_sigma="mack")
print(f"Total IBNR {mack.total_ibnr:,.0f}, standard error {mack.total_mack_se:,.0f}")
mack.summary().round(3)

In [ ]:
# Your own data, one row per payment
payments = pd.DataFrame({
    "accident_year": [2021, 2021, 2021, 2022, 2022, 2023],
    "development":   [1, 2, 3, 1, 2, 1],
    "paid":          [100.0, 50.0, 15.0, 110.0, 66.0, 120.0],
})
tri = Triangle.from_long(payments, origin="accident_year", development="development",
                         value="paid", cumulative=False)
ChainLadder(tri).summary()

## 7. When the chain-ladder fails

`diagnose` runs tests of the chain-ladder assumptions. A flag is a reason to look, not a finding: the tests raise false alarms on small triangles.

In [ ]:
from actuneo.loss_reserving import GLMReserving, diagnose, backtest, link_ratio_outliers

diagnose(raa)[["flag", "finding"]]

In [ ]:
print(link_ratio_outliers(raa))
backtest(raa).round(1)            # would the chain-ladder have predicted the latest diagonal?

In [ ]:
# The over-dispersed Poisson GLM: same reserve as the chain-ladder, with a prediction error
glm = GLMReserving(load_genins())
print(glm)
glm.summary().round(0)

A simulated portfolio in which claims inflation jumps from 3% to 40% in the last two years. The simulator knows what the claims finally cost, so each method's error can be measured.

In [ ]:
from actuneo.simulation import ClaimsSimulator, benchmark_reserving

portfolio = ClaimsSimulator(
    n_years=10, first_year=2015, claims_per_year=400, claim_cv=1.0,
    inflation=0.03, pattern_concentration=3, seed=1,
).with_inflation_shock(2022, 0.40)

triangle = portfolio.triangle()
diagnose(triangle)[["flag", "finding"]]

In [ ]:
calendar = GLMReserving(triangle, structure="calendar", trend_periods=2)
print(calendar.calendar_inflation.round(2))      # inflation estimated from the triangle

methods = {
    "Chain-ladder": ChainLadder,
    "Calendar GLM": lambda t: GLMReserving(t, structure="calendar", trend_periods=2),
}
benchmark_reserving(portfolio, methods, n_simulations=20).round(3)

## 8. IFRS 17: premium allocation approach

A group of one-year motor policies over four quarters. The premium of 1,200 is received at the start and commission of 180 is paid.

In [ ]:
from actuneo.ifrs17 import PAAGroup, PAAReinsuranceHeld, IFRS17Statements

quarters = ["Q1", "Q2", "Q3", "Q4"]

motor = PAAGroup(
    premiums_received=[1200, 0, 0, 0],
    acquisition_cash_flows=[180, 0, 0, 0],
    incurred_claims=[200, 210, 190, 220],        # claims incurred each quarter
    incurred_risk_adjustment=[10, 10, 10, 10],
    claims_paid=[100, 180, 200, 150],
    closing_lic_pv=[100, 125, 120, 185],         # reserve at each quarter end
    closing_lic_ra=[10, 14, 12, 15],
    finance_expenses=[0, 2, 3, 3],               # unwind of discount
    periods=quarters, name="Motor 2026", portfolio="Motor",
)
motor.lrc_rollforward()           # liability for remaining coverage

In [ ]:
motor.lic_rollforward()           # liability for incurred claims

In [ ]:
motor.profit_or_loss()

In [ ]:
# 20% quota share reinsurance, then the financial statements
quota_share = PAAReinsuranceHeld(
    premiums_paid=[240, 0, 0, 0],
    recoveries_incurred=[40, 42, 38, 44],
    recoveries_received=[20, 36, 40, 30],
    periods=quarters,
)

statements = IFRS17Statements(
    [motor], [quota_share], opening_equity=500,
    investment_return=[10, 12, 12, 13], other_operating_expenses=20, tax_rate=0.25,
)
statements.profit_or_loss()

In [ ]:
statements.financial_position()

In [ ]:
statements.cash_flows()

In [ ]:
statements.key_ratios().round(3)

In [ ]:
# Formatted Excel workbook (needs openpyxl)
statements.to_excel("ifrs17_statements.xlsx")

## 9. Liability for incurred claims from a reserving model

Discounted, with a risk adjustment at a chosen confidence level.

In [ ]:
from actuneo.ifrs17 import LiabilityForIncurredClaims

lic = LiabilityForIncurredClaims.from_reserving(mack, discount_rate=0.08, confidence_level=0.75)
print(lic)

## 10. The example case and blank templates

Kopje Insurance is a fictional insurer shipped with the package for demonstration. The templates are blank workbooks to fill in with your own data.

In [ ]:
from actuneo.ifrs17 import load_paa_case, example_case_path
from actuneo.templates import list_templates, save_template

case = load_paa_case(example_case_path())
print(case)
print(list_templates())

In [ ]:
for name in list_templates():
    print(save_template(name, ".", overwrite=True))